# ODIR-5K + Human Vital Signs Proxy Fusion Notebook

This notebook rebuilds the **portable** version of the earlier NeoXi proof-of-concept without any bespoke NeoXi tools.

It downloads two public Kaggle datasets, performs EDA/preprocessing, creates retinal-image records from ODIR-5K, creates 60-step vital-sign time-series windows, and builds a **proxy multimodal fusion table**.

**Important clinical caveat:** these are unrelated public datasets. The fusion is only a technical demonstration of the pipeline shape. It is **not** clinically valid for sepsis/shock prediction.

Datasets used:

- ODIR-5K / Ocular Disease Recognition: `andrewmvd/ocular-disease-recognition-odir5k`
- Human Vital Sign Dataset: `nasirayub2/human-vital-sign-dataset`

## 0. Run settings

Default mode is intentionally lightweight so a laptop can run it:

- `MAX_ODIR_PATIENTS = 1000` creates up to 2000 eye records.
- `MAX_IMAGE_FEATURES = 1000` limits image feature extraction for speed.
- `USE_TORCH_RESNET = False` uses simple PIL/numpy image features. Set to `True` for 2048-dim ResNet50 features if `torch`/`torchvision` are installed.
- Vital-sign sequence embeddings are produced with PCA to 64 dimensions by default. This preserves the shape of the original LSTM-embedding workflow while avoiding bespoke model files.

If you want to train a true LSTM later, the prepared `X_seq` / `y_seq` arrays are ready for PyTorch/Keras.

In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile, json, math, textwrap, shutil
from glob import glob

# Reproducibility / portability settings
# Resolve repo root whether this notebook is launched from the repo root or from notebooks/.
cwd = Path.cwd().resolve()
if cwd.name == "notebooks" and (cwd.parent / "README.md").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd / "README.md").exists():
    PROJECT_ROOT = cwd
else:
    PROJECT_ROOT = cwd
DATA_DIR = PROJECT_ROOT / "data" / "kaggle"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts_portable"
FIG_DIR = ARTIFACT_DIR / "figures"
PROCESSED_DIR = ARTIFACT_DIR / "processed"

for p in [DATA_DIR, ARTIFACT_DIR, FIG_DIR, PROCESSED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

ODIR_SLUG = "andrewmvd/ocular-disease-recognition-odir5k"
VITAL_SIGNS_SLUG = "nasirayub2/human-vital-sign-dataset"

MAX_ODIR_PATIENTS = 1000
MAX_IMAGE_FEATURES = 1000      # set None to process all eye images found
VITAL_SAMPLE_ROWS = 50_000     # mirrors the original workflow; set None for all rows
SEQUENCE_LENGTH = 60
PREDICTION_HORIZON = 1
VITAL_EMBED_DIM = 64
USE_TORCH_RESNET = False       # True = 2048-dim ResNet50 features if torch/torchvision installed
RANDOM_SEED = 42

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Artifacts:", ARTIFACT_DIR)

## 1. Install/import normal Python dependencies

This uses normal open-source Python libraries only. No NeoXi agents/tools are required.

Kaggle download options:

1. Preferred: `kagglehub` can download datasets by slug.
2. Fallback: Kaggle CLI (`kaggle datasets download ...`) if configured with `~/.kaggle/kaggle.json`.

If Kaggle auth is not configured, download the datasets manually from Kaggle and place them under `data/kaggle/`.

In [ ]:
def pip_install(packages):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + packages
    print("Installing:", " ".join(packages))
    subprocess.check_call(cmd)

required = ["numpy", "pandas", "matplotlib", "seaborn", "scikit-learn", "pillow", "openpyxl", "tqdm"]
missing = []
for pkg, import_name in {
    "numpy":"numpy", "pandas":"pandas", "matplotlib":"matplotlib", "seaborn":"seaborn",
    "scikit-learn":"sklearn", "pillow":"PIL", "openpyxl":"openpyxl", "tqdm":"tqdm"
}.items():
    try:
        __import__(import_name)
    except ImportError:
        missing.append(pkg)
if missing:
    pip_install(missing)

try:
    import kagglehub
except ImportError:
    try:
        pip_install(["kagglehub"])
        import kagglehub
    except Exception as e:
        kagglehub = None
        print("kagglehub unavailable; will try Kaggle CLI fallback.", repr(e))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageStat
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import f1_score, accuracy_score, mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")
np.random.seed(RANDOM_SEED)
print("Imports ready")

## 2. Download and unzip Kaggle datasets

The original repo documented the vital-sign dataset download as:

```bash
kaggle datasets download nasirayub2/human-vital-sign-dataset
```

ODIR-5K is the common Kaggle dataset:

```bash
kaggle datasets download andrewmvd/ocular-disease-recognition-odir5k
```

In [ ]:
def download_with_kagglehub(slug: str, dest: Path) -> Path:
    if kagglehub is None:
        raise RuntimeError("kagglehub is not installed/available")
    print(f"Downloading with kagglehub: {slug}")
    downloaded = Path(kagglehub.dataset_download(slug))
    dest.mkdir(parents=True, exist_ok=True)
    # Copy/symlink-like materialization so the notebook has a predictable local layout.
    for item in downloaded.iterdir():
        target = dest / item.name
        if target.exists():
            continue
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)
    return dest

def download_with_kaggle_cli(slug: str, dest: Path) -> Path:
    dest.mkdir(parents=True, exist_ok=True)
    zip_name = slug.split("/")[-1] + ".zip"
    zip_path = dest / zip_name
    cmd = ["kaggle", "datasets", "download", slug, "-p", str(dest)]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
    if zip_path.exists():
        print("Unzipping", zip_path)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(dest)
        zip_path.unlink()
    return dest

def ensure_dataset(slug: str, local_name: str) -> Path:
    dest = DATA_DIR / local_name
    if dest.exists() and any(dest.rglob("*")):
        print(f"Already present: {dest}")
        return dest
    try:
        return download_with_kagglehub(slug, dest)
    except Exception as e:
        print("kagglehub failed:", repr(e))
        try:
            return download_with_kaggle_cli(slug, dest)
        except Exception as e2:
            print("Kaggle CLI failed:", repr(e2))
            print(f"Manual fallback: download {slug} from Kaggle and extract into {dest}")
            raise

odir_dir = ensure_dataset(ODIR_SLUG, "ocular_disease_odir5k")
vital_dir = ensure_dataset(VITAL_SIGNS_SLUG, "human_vital_sign_dataset")
print("ODIR dir:", odir_dir)
print("Vital signs dir:", vital_dir)

## 3. Locate dataset files robustly

Kaggle archives sometimes nest files differently. These helper functions search for the files the original workflows expected:

- ODIR metadata: `data.xlsx`
- ODIR images: `Training Images/`
- Vital signs CSV: `human_vital_signs_dataset_2024.csv`

In [ ]:
def find_one(root: Path, patterns, required=True):
    if isinstance(patterns, str):
        patterns = [patterns]
    hits = []
    for pat in patterns:
        hits.extend(root.rglob(pat))
    hits = sorted(set(hits), key=lambda p: (len(p.parts), str(p)))
    if not hits and required:
        raise FileNotFoundError(f"No match under {root} for patterns: {patterns}")
    return hits[0] if hits else None

odir_xlsx = find_one(odir_dir, "data.xlsx")
training_dirs = [p for p in odir_dir.rglob("Training Images") if p.is_dir()]
if not training_dirs:
    raise FileNotFoundError("Could not find ODIR 'Training Images' directory")
odir_train_img_dir = sorted(training_dirs, key=lambda p: len(p.parts))[0]

vital_csv = find_one(vital_dir, ["human_vital_signs_dataset_2024.csv", "*.csv"])

print("ODIR metadata:", odir_xlsx)
print("ODIR training images:", odir_train_img_dir)
print("Vital signs CSV:", vital_csv)
print("ODIR image count:", len(list(odir_train_img_dir.glob("*.jpg"))))

## 4. ODIR-5K EDA and preprocessing

The NeoXi workflow did roughly this:

1. Load `data.xlsx`.
2. Take the first 1000 patients.
3. Parse multi-label disease columns: `N, D, G, C, A, H, M, O`.
4. Create one row per eye with image filename/path.
5. Later extract image features.

In [ ]:
odir = pd.read_excel(odir_xlsx)
print("ODIR shape:", odir.shape)
display(odir.head())
print(odir.dtypes)

label_cols = [c for c in ["N", "D", "G", "C", "A", "H", "M", "O"] if c in odir.columns]
print("Label columns:", label_cols)

fig, ax = plt.subplots(figsize=(8, 4))
odir[label_cols].sum().sort_values(ascending=False).plot(kind="bar", ax=ax, title="ODIR label counts")
ax.set_ylabel("patients")
plt.tight_layout()
plt.show()

if "Patient Age" in odir.columns:
    odir["Patient Age"].hist(bins=30, figsize=(7, 3))
    plt.title("ODIR patient age distribution")
    plt.xlabel("age")
    plt.ylabel("count")
    plt.show()

In [ ]:
def make_odir_eye_records(df: pd.DataFrame, image_dir: Path, max_patients=1000) -> pd.DataFrame:
    df = df.head(max_patients).copy()
    records = []
    for _, row in df.iterrows():
        labels = [int(row.get(c, 0)) for c in ["N", "D", "G", "C", "A", "H", "M", "O"]]
        for eye, filename_col, keyword_col in [
            ("left", "Left-Fundus", "Left-Diagnostic Keywords"),
            ("right", "Right-Fundus", "Right-Diagnostic Keywords"),
        ]:
            filename = row.get(filename_col)
            if pd.isna(filename) or not str(filename).strip():
                continue
            image_path = image_dir / str(filename)
            records.append({
                "ID": row.get("ID"),
                "Patient Age": row.get("Patient Age"),
                "Patient Sex": row.get("Patient Sex"),
                "eye": eye,
                "image_filename": str(filename),
                "image_path": str(image_path),
                "diagnostic_keywords": row.get(keyword_col, ""),
                "labels": labels,
                **{c: int(row.get(c, 0)) for c in ["N", "D", "G", "C", "A", "H", "M", "O"] if c in row.index}
            })
    out = pd.DataFrame(records)
    out["image_exists"] = out["image_path"].map(lambda p: Path(p).exists())
    return out

odir_records = make_odir_eye_records(odir, odir_train_img_dir, MAX_ODIR_PATIENTS)
print("Eye records:", odir_records.shape)
print("Images exist:", odir_records["image_exists"].mean())
display(odir_records.head())

odir_records.to_csv(PROCESSED_DIR / "odir5k_eye_manifest.csv", index=False)
print("Saved:", PROCESSED_DIR / "odir5k_eye_manifest.csv")

## 5. Portable retinal image features

The original project extracted **ResNet50 2048-dim features** plus additional classical image features. To keep this notebook runnable anywhere, the default extractor uses simple PIL/numpy features:

- RGB mean/std/min/max
- small RGB histograms
- grayscale mean/std and quantiles

If you set `USE_TORCH_RESNET = True`, the notebook will try to use `torchvision.models.resnet50` and output 2048-dim embeddings.

In [ ]:
def simple_image_features(path, size=(224, 224), bins=16):
    path = Path(path)
    if not path.exists():
        return None
    try:
        img = Image.open(path).convert("RGB").resize(size)
        arr = np.asarray(img).astype("float32") / 255.0
        feats = []
        # RGB channel summaries
        for ch in range(3):
            x = arr[:, :, ch].ravel()
            feats.extend([x.mean(), x.std(), x.min(), x.max()])
            hist, _ = np.histogram(x, bins=bins, range=(0, 1), density=True)
            feats.extend(hist.tolist())
        gray = arr.mean(axis=2).ravel()
        feats.extend([gray.mean(), gray.std(), *np.quantile(gray, [0.05, 0.25, 0.5, 0.75, 0.95]).tolist()])
        return np.array(feats, dtype="float32")
    except Exception:
        return None

def resnet50_features(paths):
    import torch
    import torchvision.models as models
    import torchvision.transforms as T
    from torch.utils.data import DataLoader, Dataset
    device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights)
    model.fc = torch.nn.Identity()
    model.eval().to(device)
    transform = weights.transforms()
    feats = []
    with torch.no_grad():
        for p in tqdm(paths, desc="ResNet50 features"):
            img = Image.open(p).convert("RGB")
            x = transform(img).unsqueeze(0).to(device)
            y = model(x).cpu().numpy()[0]
            feats.append(y.astype("float32"))
    return feats

feature_records = odir_records[odir_records["image_exists"]].copy()
if MAX_IMAGE_FEATURES is not None:
    feature_records = feature_records.head(MAX_IMAGE_FEATURES).copy()

if USE_TORCH_RESNET:
    features = resnet50_features(feature_records["image_path"].tolist())
else:
    features = [simple_image_features(p) for p in tqdm(feature_records["image_path"], desc="Simple image features")]

valid = [f is not None for f in features]
feature_records = feature_records.loc[valid].copy()
features = [f for f in features if f is not None]

image_feature_matrix = np.vstack(features)
print("Image feature matrix:", image_feature_matrix.shape)

image_feature_cols = [f"image_feature_{i}" for i in range(image_feature_matrix.shape[1])]
image_features_df = pd.concat([
    feature_records.reset_index(drop=True)[["ID", "Patient Age", "Patient Sex", "eye", "image_filename", "image_path"] + label_cols],
    pd.DataFrame(image_feature_matrix, columns=image_feature_cols)
], axis=1)
image_features_df["fusion_id"] = np.arange(len(image_features_df))
image_features_df.to_pickle(PROCESSED_DIR / "odir5k_image_features.pkl")
image_features_df.to_csv(PROCESSED_DIR / "odir5k_image_features.csv", index=False)
print("Saved image features:", PROCESSED_DIR / "odir5k_image_features.pkl")

## 6. Vital signs EDA and preprocessing

The original workflow did roughly this:

1. Load `human_vital_signs_dataset_2024.csv`.
2. Sample/limit to about 50k rows.
3. Convert `Timestamp`, sort by time.
4. Drop high-collinearity derived columns (`Derived_Pulse_Pressure`, `Derived_MAP`) and patient ID.
5. Encode categorical columns (`Gender`, `Risk Category`).
6. Standardize numeric features.
7. Create 60-timestep windows for time-series forecasting.

In [ ]:
vital = pd.read_csv(vital_csv)
print("Vital shape:", vital.shape)
display(vital.head())
print(vital.dtypes)

if VITAL_SAMPLE_ROWS is not None and len(vital) > VITAL_SAMPLE_ROWS:
    vital = vital.sample(VITAL_SAMPLE_ROWS, random_state=RANDOM_SEED).copy()
    print("Sampled vital rows:", vital.shape)

# Basic EDA
print("Missing values:")
display(vital.isna().sum().sort_values(ascending=False).head(20))

display(vital.describe(include="all").T.head(30))

num_cols_raw = vital.select_dtypes(include=[np.number]).columns.tolist()
if num_cols_raw:
    corr = vital[num_cols_raw].corr(numeric_only=True)
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title("Vital signs numeric correlation")
    plt.tight_layout()
    plt.show()

In [ ]:
def preprocess_vital_signs(df: pd.DataFrame):
    out = df.copy()
    # Drop known problematic / non-model columns from original workflow if present.
    drop_cols = [c for c in ["Patient ID", "Derived_Pulse_Pressure", "Derived_MAP"] if c in out.columns]
    out = out.drop(columns=drop_cols)

    if "Timestamp" in out.columns:
        out["Timestamp"] = pd.to_datetime(out["Timestamp"], errors="coerce")
        out = out.sort_values("Timestamp").reset_index(drop=True)

    # Encode categoricals.
    encoders = {}
    for col in out.select_dtypes(include=["object", "category"]).columns:
        if col == "Timestamp":
            continue
        le = LabelEncoder()
        out[col] = le.fit_transform(out[col].astype(str))
        encoders[col] = le

    # Coerce remaining non-timestamp columns numeric where possible.
    for col in out.columns:
        if col != "Timestamp":
            out[col] = pd.to_numeric(out[col], errors="coerce")

    # Interpolate/fill numeric gaps.
    numeric_cols = [c for c in out.columns if c != "Timestamp" and pd.api.types.is_numeric_dtype(out[c])]
    out[numeric_cols] = out[numeric_cols].interpolate(limit_direction="both").ffill().bfill()
    return out, encoders

vital_processed, vital_encoders = preprocess_vital_signs(vital)
print("Processed vital shape:", vital_processed.shape)
display(vital_processed.head())

vital_processed.to_csv(PROCESSED_DIR / "vital_signs_processed_data.csv", index=False)
print("Saved:", PROCESSED_DIR / "vital_signs_processed_data.csv")

## 7. Create 60-step time-series windows and 64-dim vital embeddings

The original bespoke workflow trained/used a PyTorch LSTM and extracted 64-dim hidden embeddings. Here we create the same interface portably:

- `X_seq`: `(n_windows, 60, n_features)`
- `y_seq`: next-step `Heart Rate`
- `vital_embeddings`: 64-dim PCA embeddings from flattened standardized windows

This gives downstream users a runnable proxy. They can replace PCA with an LSTM encoder later.

In [ ]:
def create_sequences(df, target_col="Heart Rate", sequence_length=60, prediction_horizon=1):
    if target_col not in df.columns:
        raise KeyError(f"Target column {target_col!r} not found. Columns: {df.columns.tolist()}")
    feature_cols = [c for c in df.columns if c != "Timestamp" and c != target_col]
    model_cols = feature_cols + [target_col]

    values = df[model_cols].astype("float32").values
    scaler = StandardScaler()
    values_scaled = scaler.fit_transform(values).astype("float32")

    target_idx = model_cols.index(target_col)
    X, y = [], []
    max_i = len(values_scaled) - sequence_length - prediction_horizon + 1
    for i in range(max_i):
        X.append(values_scaled[i:i+sequence_length, :])
        y.append(values_scaled[i+sequence_length+prediction_horizon-1, target_idx])
    return np.stack(X), np.array(y, dtype="float32"), model_cols, scaler

X_seq, y_seq, vital_model_cols, vital_scaler = create_sequences(
    vital_processed,
    target_col="Heart Rate",
    sequence_length=SEQUENCE_LENGTH,
    prediction_horizon=PREDICTION_HORIZON,
)
print("X_seq:", X_seq.shape, "y_seq:", y_seq.shape)
print("Model columns:", vital_model_cols)

flat = X_seq.reshape(X_seq.shape[0], -1)
n_components = min(VITAL_EMBED_DIM, flat.shape[0], flat.shape[1])
pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
vital_embeddings = pca.fit_transform(flat).astype("float32")
if n_components < VITAL_EMBED_DIM:
    vital_embeddings = np.pad(vital_embeddings, ((0,0),(0,VITAL_EMBED_DIM-n_components)))
print("Vital embeddings:", vital_embeddings.shape)
print("Explained variance in PCA components:", float(np.sum(pca.explained_variance_ratio_)))

np.save(PROCESSED_DIR / "vital_X_seq.npy", X_seq)
np.save(PROCESSED_DIR / "vital_y_seq.npy", y_seq)
np.save(PROCESSED_DIR / "vital_embeddings_64.npy", vital_embeddings)

## 8. Build the proxy multimodal fusion table

This mirrors the original proof-of-concept fusion step:

- Retinal features: image feature vector per ODIR eye image.
- Vital features: 64-dim time-series embedding per vital-sign window.
- Match by sequential `fusion_id` only.

Again: sequential pairing is only for validating the mechanics of the pipeline, **not for clinical inference**.

In [ ]:
n = min(len(image_features_df), len(vital_embeddings))
image_part = image_features_df.head(n).reset_index(drop=True).copy()
vital_part = pd.DataFrame(vital_embeddings[:n], columns=[f"vital_embed_{i}" for i in range(VITAL_EMBED_DIM)])
vital_part["fusion_id"] = np.arange(n)

fusion = pd.concat([image_part.reset_index(drop=True), vital_part.drop(columns=["fusion_id"]).reset_index(drop=True)], axis=1)
fusion["fusion_id"] = np.arange(n)

print("Fusion shape:", fusion.shape)
display(fusion.head())

fusion_path = PROCESSED_DIR / "odir5k_vital_signs_proxy_fusion.csv"
fusion.to_csv(fusion_path, index=False)
print("Saved fusion table:", fusion_path)
print("Rows:", len(fusion), "Image feature dims:", len(image_feature_cols), "Vital dims:", VITAL_EMBED_DIM)

## 9. Optional sanity-check models

These are not clinical models. They only verify that the preprocessed arrays are usable by standard ML libraries.

- Vital-only next-step heart-rate regressor
- ODIR image-feature multilabel classifier

The real clinical question would require matched neonatal fundus images, synchronized NICU vitals, and shock/sepsis labels.

In [ ]:
# Vital next-step proxy regression sanity check
X_flat = X_seq.reshape(X_seq.shape[0], -1)
if len(X_flat) > 10000:
    idx = np.random.RandomState(RANDOM_SEED).choice(len(X_flat), 10000, replace=False)
    X_model, y_model = X_flat[idx], y_seq[idx]
else:
    X_model, y_model = X_flat, y_seq

X_train, X_test, y_train, y_test = train_test_split(X_model, y_model, test_size=0.2, random_state=RANDOM_SEED)
reg = RandomForestRegressor(n_estimators=50, random_state=RANDOM_SEED, n_jobs=-1, max_depth=10)
reg.fit(X_train, y_train)
pred = reg.predict(X_test)
print("Vital proxy regression MAE:", mean_absolute_error(y_test, pred))
print("Vital proxy regression RMSE:", mean_squared_error(y_test, pred) ** 0.5)

# ODIR multilabel sanity check: quick per-label RandomForest on image features.
if label_cols and len(image_features_df) >= 50:
    X_img = image_features_df[image_feature_cols].values
    print("\nODIR proxy image-feature label checks:")
    for label in label_cols:
        y = image_features_df[label].astype(int).values
        if len(np.unique(y)) < 2:
            continue
        Xtr, Xte, ytr, yte = train_test_split(X_img, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y if min(np.bincount(y)) > 1 else None)
        clf = RandomForestClassifier(n_estimators=50, random_state=RANDOM_SEED, n_jobs=-1, max_depth=10, class_weight="balanced")
        clf.fit(Xtr, ytr)
        yp = clf.predict(Xte)
        print(label, "accuracy=", round(accuracy_score(yte, yp), 3), "f1=", round(f1_score(yte, yp, zero_division=0), 3))

## 10. What to replace for a real NICU study

For real shock/sepsis prediction, replace the public proxy inputs with matched clinical data:

1. Neonatal fundus images with timestamps.
2. NICU vital signs at 1-minute resolution or similar.
3. Clinically adjudicated shock/sepsis labels at prediction times.
4. Matching logic by patient identifier and timestamp window, not sequential `fusion_id`.
5. Patient-level train/test split to prevent leakage.
6. Evaluation: sensitivity, specificity, AUROC/AUPRC, calibration, and clinician-reviewed explanations.

The reusable artifacts from this notebook are the *data contract* and *pipeline skeleton*, not the proxy labels.